# Lab 1 — Serving under load: watch the death spiral

**The claim you should be able to make when you finish:** *"An LLM server under
overload does not fail by returning errors. It fails by queueing, and every
request gets slower until none of them are useful. I have watched it happen and
I can show you which three metrics said so first."*

That is the interview answer. This notebook is how you earn the right to say it.

### What you will do

1. Predict, on paper, how much concurrency this GPU can hold — before starting anything.
2. Start vLLM as a background process and scrape its `/metrics`.
3. Drive it at a load it can serve, and plot queue depth, KV-cache use, TTFT live.
4. Drive it past capacity and watch the same three lines diverge.
5. Show that throughput stayed flat while **goodput** went to zero. That gap is the lab.

### The notebook as a dashboard

A serving lab in a notebook sounds like a compromise. It is not: the server runs
detached with its log on disk, cells poll `/metrics` every second, and one cell
live-updates a matplotlib chart. Watching the spiral develop as a chart is
better intuition than watching it scroll past in a terminal.

What a notebook *cannot* rehearse is the feeling of diagnosing this live, which
is what an interview actually tests. Do that part in `terminal/` once these
cells work — tmux, curl, and the log, no cells.

In [ ]:
# Cell 1 — everything that must survive a disconnect. Idempotent: rerun freely.
#
# Colab sessions are ephemeral. Keep every install and download here so a
# dropped runtime costs three minutes, not the evening.

REPO   = "https://github.com/lsgrep/serv.git"
BRANCH = "main"

import os, subprocess, sys

if not os.path.isdir("serv"):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO], check=True)
else:
    subprocess.run(["git", "-C", "serv", "pull", "--ff-only", "-q"], check=False)
sys.path.insert(0, os.path.abspath("serv"))

In [ ]:
# vLLM pulls its own torch build, so this is the slow cell (5-10 min, once).
#
# If Colab asks you to restart the session afterwards: do it, then rerun cell 1
# and this cell. The second run is a no-op.
#
# VLLM_PIN: leave empty for the current release. If you hit kernel errors on a
# T4 mentioning sm_75 or bf16, pin an older release here — recent vLLM
# increasingly assumes Ampere or newer, and that is the hardware, not you.
VLLM_PIN = ""

import subprocess, sys

def pip(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=True)

pip("matplotlib", "pandas", "httpx")
pip(f"vllm=={VLLM_PIN}" if VLLM_PIN else "vllm")

import servlab
env = servlab.notebook_setup()

## 1. Predict before you measure

Do not start the server yet. Work out what it should be able to do, so the
measurement has something to disagree with.

The one number everything follows from:

$$\text{KV bytes per token} = 2 \times \text{layers} \times \text{kv\_heads} \times \text{head\_dim} \times \text{bytes}$$

The leading 2 is K and V. `kv_heads` (not `heads`) is the GQA discount: Llama-3
runs 32 query heads over 8 KV heads, so its cache is 4x smaller than the naive
calculation suggests. Getting this wrong by 4x is the most common napkin-math
error in an interview.

In [ ]:
from servlab import napkin as nk

# A 3B model in fp16 is the right size for a 16 GB card: ~6 GiB of weights
# leaves ~7-8 GiB of KV, which is enough concurrency for queueing to be the
# interesting failure rather than OOM. An 8B in fp16 does not fit at all —
# print the report for it too and see the engine tell you so.
MODEL = "Qwen/Qwen2.5-3B-Instruct"
SPEC  = nk.MODELS["qwen2.5-3b"]
GPU   = "T4" if "T4" in env.gpu_name else ("L4" if "L4" in env.gpu_name else "A100-40GB")

MAX_MODEL_LEN = 2048          # cap context so the math is easy to reason about
GPU_UTIL      = 0.90

print(nk.memory_report(GPU, SPEC, seq_len=MAX_MODEL_LEN, util=GPU_UTIL))
print()
print(nk.memory_report(GPU, "llama-3.1-8b", seq_len=MAX_MODEL_LEN, util=GPU_UTIL))

In [ ]:
# Now turn memory into a load ceiling.
#
# Concurrency ceiling comes from memory. Throughput ceiling comes from
# bandwidth: decode is memory bound, so a step costs (weights + KV) / bandwidth.

TYPICAL_PROMPT = 256
TYPICAL_OUTPUT = 128
CTX = TYPICAL_PROMPT + TYPICAL_OUTPUT

max_seqs = nk.max_concurrent_sequences(GPU, SPEC, seq_len=CTX, util=GPU_UTIL)
step_s   = nk.decode_step_time_s(GPU, SPEC, batch=16, ctx_len=CTX)
tok_s    = nk.decode_tokens_per_s(GPU, SPEC, batch=16, ctx_len=CTX)

# Time to serve one request end to end, at a batch the server will actually run.
service_s = nk.prefill_time_s(GPU, SPEC, TYPICAL_PROMPT) + TYPICAL_OUTPUT * step_s
capacity_rps = nk.saturation_rps(service_s / 16)   # 16 in flight at once

print(f"KV per token       {nk.human_bytes(nk.kv_bytes_per_token(SPEC))}")
print(f"concurrency ceiling {max_seqs:,.0f} sequences @ {CTX} tokens")
print(f"decode step (b=16)  {step_s*1000:.1f} ms  ->  {tok_s:,.0f} output tok/s")
print(f"service time        {service_s:.2f} s per request")
print(f"capacity            ~{capacity_rps:.1f} req/s before the queue grows")
print()
print("Write these down. The measurement will disagree; the gap is the lesson.")

In [ ]:
# Why the last 10% of utilisation is the expensive part.
#
# This is M/M/1 — not a model of vLLM, a model of why queueing is nonlinear.
# Nothing about the server changes between 80% and 95% load. The wait quadruples.
import matplotlib.pyplot as plt
from servlab.plots import use_style, SERIES, STATUS

use_style()
loads = [i / 100 for i in range(50, 100)]
waits = [nk.mm1_wait_s(rho * capacity_rps, service_s / 16) for rho in loads]

fig, ax = plt.subplots(figsize=(7.5, 4))
ax.plot([r * 100 for r in loads], waits, color=SERIES[0])
ax.axvline(80, color=STATUS["warning"], linestyle=":", linewidth=1.5)
ax.annotate("80%", xy=(80, max(waits) * 0.5), xytext=(4, 0),
            textcoords="offset points", color=STATUS["warning"], fontsize=9)
ax.set_xlabel("offered load (% of capacity)")
ax.set_ylabel("expected queue wait (s)")
ax.set_title("queue wait vs utilisation — the wall is not where the capacity is")
plt.show()

## 2. Start the server in the background

`nohup vllm serve ... &` in a shell cell works, but `VLLMServer` does the same
thing and adds the parts you would otherwise write twice: it forces
`--dtype half` on pre-Ampere cards (Turing has no bf16, and vLLM reads
`bfloat16` straight out of most modern configs, so it refuses to start), it
detaches the process so a `KeyboardInterrupt` in a cell does not kill it, and it
prints the tail of the log when startup fails.

Startup takes a few minutes the first time — the weights have to download.

In [ ]:
from servlab.serve import VLLMServer

server = VLLMServer(
    MODEL,
    port=8000,
    max_model_len=MAX_MODEL_LEN,
    gpu_memory_utilization=GPU_UTIL,
    enforce_eager=True,        # skip CUDA-graph capture: faster start, more free VRAM
    log_path="runs/lab1.log",
)
server.start()          # blocks until /health is 200; raises with the log on failure

In [ ]:
# What did the engine actually decide? These two lines from the log are the
# ground truth your napkin math was guessing at.
print("\n".join(l for l in server.tail(200).splitlines()
                 if "KV cache" in l or "concurrency" in l or "Maximum" in l) or server.tail(20))

In [ ]:
# One request, to prove the endpoint works and to see the shape of the response.
from servlab.loadgen import run_load
from servlab.stats import summarize

warm = run_load("http://localhost:8000", MODEL, concurrency=1, n_requests=2,
                duration=None, prompt_tokens=64, max_tokens=32)
print(summarize(warm))
print("\nsample output:", warm[-1].text[:120].replace("\n", " "))

In [ ]:
# The metrics endpoint, parsed. These are the seven numbers the dashboard plots.
from servlab.monitor import scrape_row

for k, v in scrape_row("http://localhost:8000").items():
    print(f"  {k:<16} {v}")

## 3. A load it can serve

Start the poller, then run load in a background thread, then watch the live
chart. Three cells, run in order, without waiting for the load to finish.

**Closed loop vs open loop matters more than any other choice here.** With
`concurrency=N`, each worker sends its next request only when the previous one
returns — so when the server slows down, the client offers *less* load, and the
server can never be overloaded. That is fine for measuring peak throughput and
useless for reproducing an outage. With `rps=R`, requests launch on a Poisson
schedule whether or not the server is keeping up. Real traffic is open loop.

A benchmark that shows a server degrading gracefully forever was closed loop.

In [ ]:
import threading
from servlab.monitor import MetricsPoller

BASE = "http://localhost:8000"
poller = MetricsPoller(BASE, interval=1.0).start()

calm = {}
def calm_run():
    calm["results"] = run_load(BASE, MODEL, rps=1.0, duration=90,
                               prompt_tokens=TYPICAL_PROMPT, max_tokens=TYPICAL_OUTPUT)

t = threading.Thread(target=calm_run, daemon=True)
t.start()
print("load started in the background — run the next cell now")

In [ ]:
# The notebook as Grafana. Redraws in place once a second.
from servlab.plots import live_dashboard

live_dashboard(poller, seconds=95, slo_ttft=1.0, title="lab 1 — within capacity")

In [ ]:
t.join()
calm_summary = summarize(calm["results"], slo_ttft=1.0, slo_tpot=0.05)
print(calm_summary)
calm_rows = list(poller.rows)
poller.stop()

Read the chart before moving on:

* **waiting** should sit at or near zero. Work arrives and is served immediately.
* **KV cache %** plateaus well under 100. Memory is not the constraint yet.
* **TTFT p50 and p99 are close together.** A tight spread means nobody is queueing.

If `waiting` is already climbing at 1 req/s, your prediction was wrong by a lot —
check `--max-num-seqs` in the log and what the engine says its KV capacity is.

## 4. Now overload it

Same workload, same server, three to four times the arrival rate — open loop, so
the client will not back off. Nothing is misconfigured; there is simply more work
arriving than leaving.

Watch the order in which things go wrong. It is always the same, and being able
to recite it is worth more than any single number:

1. `waiting` starts to climb — the queue is the first thing to move.
2. `kv_cache` pins at ~100% — every block is spoken for.
3. **preemptions** begin — the engine evicts a running sequence to admit another,
   throwing away its KV cache. In recompute mode that work is done twice.
4. TTFT p99 detaches from p50, then p50 follows it up.
5. Throughput stays flat. **This is the part people get wrong.** The server is
   as busy as it ever was. It is just that nobody is getting an answer in time.

In [ ]:
poller = MetricsPoller(BASE, interval=1.0).start()

OVERLOAD_RPS = 4.0     # raise until `waiting` climbs without bound

hot = {}
def hot_run():
    hot["results"] = run_load(BASE, MODEL, rps=OVERLOAD_RPS, duration=90,
                              prompt_tokens=TYPICAL_PROMPT, max_tokens=TYPICAL_OUTPUT)

t2 = threading.Thread(target=hot_run, daemon=True)
t2.start()
print("overload started — run the next cell now")

In [ ]:
live_dashboard(poller, seconds=100, slo_ttft=1.0, title="lab 1 — past capacity")

In [ ]:
t2.join()
hot_summary = summarize(hot["results"], slo_ttft=1.0, slo_tpot=0.05)
print(hot_summary)
hot_rows = list(poller.rows)
poller.stop()

## 5. The gap between throughput and goodput

Throughput counts requests that finished. Goodput counts requests that finished
*and were still useful* — inside the latency SLO. A saturated server holds
throughput almost perfectly flat while goodput collapses to zero, which is
exactly why a throughput dashboard will tell you everything is fine during an
outage.

In [ ]:
from servlab.plots import bar_compare, latency_cdf

print(f"{'':<12}{'req/s':>10}{'out tok/s':>12}{'TTFT p50':>12}{'TTFT p99':>12}{'goodput':>10}")
for name, s in (("calm", calm_summary), ("overload", hot_summary)):
    print(f"{name:<12}{s.request_throughput:>10.2f}{s.output_throughput:>12,.0f}"
          f"{s.ttft['p50']*1000:>11,.0f}ms{s.ttft['p99']*1000:>11,.0f}ms{s.goodput:>10.2f}")

fig, ax = bar_compare(
    ["calm\nthroughput", "calm\ngoodput", "overload\nthroughput", "overload\ngoodput"],
    [calm_summary.request_throughput, calm_summary.goodput,
     hot_summary.request_throughput, hot_summary.goodput],
    title="throughput held; goodput did not", ylabel="requests / s",
    highlight={"overload\ngoodput"}, fmt="{:,.2f}")

In [ ]:
latency_cdf({"within capacity": [r.ttft for r in calm["results"] if r.ok],
             "past capacity":   [r.ttft for r in hot["results"] if r.ok]},
            title="TTFT distribution — the whole curve moved, not just the tail",
            slo=1.0)

In [ ]:
# Both runs on one time axis, so the sequence of failures is legible.
from servlab.plots import dashboard

offset = calm_rows[-1]["t"] if calm_rows else 0
combined = calm_rows + [{**r, "t": r["t"] + offset} for r in hot_rows]
dashboard(combined, title="lab 1 — 1 req/s for 90s, then 4 req/s", slo_ttft=1.0);

## 6. What you would actually do about it

Having watched it, you can now answer the follow-up question, which is the one
that separates candidates. Rough order of cheapness:

| Lever | What it does | What it costs |
|---|---|---|
| **Admission control / rate limit** | Reject or shed past a queue depth | Some requests fail fast — usually better than everyone timing out |
| **Cap `--max-num-seqs`** | Bounds in-flight work, so latency is bounded too | Lower peak throughput |
| **Shorter `--max-model-len`** | More KV blocks per GB, more concurrency | Long prompts rejected |
| **FP8 / INT8 KV cache** | Halves KV bytes per token, doubles concurrency | Small quality risk (lab 5) |
| **Chunked prefill** | Stops a long prefill from stalling everyone's decode | Slightly worse TTFT for the long request |
| **Prefix caching** | Shared system prompts are prefilled once | Only helps if traffic shares prefixes |
| **More GPUs** | The honest answer when the above is spent | Money (lab 2) |

The failure mode to name out loud: **queueing, not errors.** A server that
returns 503 under overload is behaving well. A server that accepts everything
and slows down is the one that takes the product with it.

### Prove the fix

Restart with a hard cap on in-flight sequences and rerun the same overload. TTFT
should stop climbing — bounded work in flight means bounded latency. What was
unbounded before is now the queue in front of the server, which is a queue you
can see, rate-limit, and shed.

In [ ]:
server.stop()

capped = VLLMServer(MODEL, port=8000, max_model_len=MAX_MODEL_LEN,
                    gpu_memory_utilization=GPU_UTIL, enforce_eager=True,
                    max_num_seqs=8, log_path="runs/lab1-capped.log").start()

poller = MetricsPoller(BASE, interval=1.0).start()
capped_results = run_load(BASE, MODEL, rps=OVERLOAD_RPS, duration=60,
                          prompt_tokens=TYPICAL_PROMPT, max_tokens=TYPICAL_OUTPUT)
poller.stop()

capped_summary = summarize(capped_results, slo_ttft=1.0, slo_tpot=0.05)
print(capped_summary)
print(f"\nTTFT p99: {hot_summary.ttft['p99']:.1f}s uncapped  ->  "
      f"{capped_summary.ttft['p99']:.1f}s with max_num_seqs=8")
dashboard(list(poller.rows), title="same overload, max_num_seqs=8", slo_ttft=1.0);

In [ ]:
# Free the GPU. Colab will not do it for you, and lab 2 needs the VRAM.
capped.stop()

## 7. If you have no GPU right now

The dynamics are not actually about CUDA. `servlab.toy` runs the same scheduler
— admission control, paged blocks, preemption — against a virtual clock, and
reproduces the spiral in a few seconds of CPU time. Useful on a plane, and
useful for building intuition about *which knob does what* before spending GPU
minutes on it.

In [ ]:
from servlab.toy.scheduler import simulate
from servlab.plots import dashboard

rows_calm, _ = simulate(rps=1.5,  duration=60, num_blocks=512)
rows_hot,  _ = simulate(rps=14.0, duration=60, num_blocks=512)

dashboard(rows_calm, title="simulated — within capacity")
dashboard(rows_hot,  title="simulated — past capacity: queue grows, preemption starts");

## Take it to the terminal

You now have the intuition. The interview does not feel like this — it feels
like a terminal with a log scrolling and someone watching you think.

`terminal/` in this repo is the same lab as shell scripts: tmux with four panes
(server log, `watch` on `/metrics`, load generator, scratch shell), plus
`terminal/DIAGNOSIS.md`, which is a set of scenarios to work through out loud.
Run it once on a rented box after this notebook works.

**Next:** lab 2 takes this sweep and runs it on three different GPUs.